# Домашнее задание. Английское эссе и именованные сущности

**Мотыгуллин Булат Габдулхакович**  
**Группа: Б01-301г**

In [1]:
from course import *

seed_all(11)
OUT = ROOT / "results" / "92_summer_entities"
OUT.mkdir(exist_ok=True, parents=True)

## Английское сочинение

Текст хранится в summer_essay.txt.

In [2]:
essay = (ROOT / "summer_essay.txt").read_text()
display(Markdown(essay.replace("\n\n", "\n\n")))
print("Число слов:", len(essay.split()))

How I Spent My Summer

My summer in 2026 combined programming, mathematics, and time with friends. I spent most of it in Moscow and Dolgoprudny, where I could keep working on my projects without following a strict university timetable. As a student at the Moscow Institute of Physics and Technology, I wanted to use the break to understand several ideas that had seemed too abstract during the semester.

In June 2026, I organised my study materials in Obsidian. My notes had gradually grown into a collection of formulas, programming examples, and unfinished questions. I connected related topics and made a more convenient structure for machine learning and information theory. Two classmates joined me for several study sessions in Dolgoprudny. We discussed difficult exercises, compared our explanations, and sometimes discovered that a small numerical example was more useful than another page of theory.

During July 2026, I spent more time on Python and PyTorch. I trained small neural networks and looked carefully at their errors. Changing a learning rate or an activation function often produced a surprisingly different result. I also read about large language models and the reuse of cached intermediate computations. This topic became interesting because it connects the quality of an answer with the time and memory required to obtain it. I used GitHub to explore public implementations and kept a list of experiments that I would like to reproduce.

Another part of the summer was devoted to optimisation problems. I explored how software can place objects, build routes, and distribute limited resources. These projects made mathematical models feel practical. A complicated formula became easier to understand when I could change an input and see how the solution reacted. Discussing these examples with friends also helped me explain my ideas more clearly.

In August 2026, I tried to balance this work with ordinary summer life in Moscow. I met friends for walks and left some evenings free from coding. Returning to a problem the next morning often helped more than continuing to debug it late at night. By the end of the summer, I had a better-organised collection of notes, several small experiments, and a clearer idea of the subjects I wanted to study further. The most useful result was a habit of checking an explanation with a reproducible example.


Число слов: 382


## spaCy: исходный NER и уточнение специальных названий

Использую en_core_web_sm. Сначала вывожу исходные сущности, затем добавляю правила для названий приложений и библиотек. GPE — место, ORG — организация, PRODUCT — продукт, LANGUAGE — язык.

In [3]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")
original = nlp(essay)
table(
    [
        {"entity": e.text, "type": e.label_, "start": e.start_char, "end": e.end_char}
        for e in original.ents
    ],
    OUT,
    "spacy_original_entities",
)
ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
labels = {
    "Moscow": "GPE",
    "Dolgoprudny": "GPE",
    "Moscow Institute of Physics and Technology": "ORG",
    "Obsidian": "PRODUCT",
    "Python": "LANGUAGE",
    "PyTorch": "PRODUCT",
    "GitHub": "ORG",
    "June 2026": "DATE",
    "July 2026": "DATE",
    "August 2026": "DATE",
    "2026": "DATE",
}
ruler.add_patterns(
    [{"label": label, "pattern": text} for text, label in labels.items()]
)
doc = nlp(essay)
rows = [
    {
        "entity": e.text,
        "type": e.label_,
        "meaning": spacy.explain(e.label_),
        "start": e.start_char,
        "end": e.end_char,
    }
    for e in doc.ents
]
df = table(rows, OUT, "all_named_entities")
(OUT / "entities_highlighted.html").write_text(
    displacy.render(doc, style="ent", page=True, jupyter=False), encoding="utf-8"
)
display(df[["entity", "type"]].drop_duplicates().reset_index(drop=True))
note(
    "Именованные сущности — конкретные названия, даты и количества. Обычные слова friends, classmates и neural networks сами по себе именованными сущностями не являются. Английская статистическая модель может пропускать технические названия, поэтому финальный результат дополнен явно приведёнными правилами. В тексте есть место, даты и описание совместной работы с одногруппниками."
)

,entity,type,start,end
0,summer,DATE,26,32
1,2026,DATE,36,40
2,Moscow,GPE,121,127
3,the Moscow Institute of Physics and Technology,ORG,252,298
4,June 2026,DATE,408,417
5,Obsidian,GPE,453,461
6,Two,CARDINAL,680,683
7,Dolgoprudny,GPE,735,746
8,July 2026,DATE,918,927
9,PyTorch,PERSON,961,968


,entity,type,meaning,start,end
0,summer,DATE,Absolute or relative dates or periods,26,32
1,2026,DATE,Absolute or relative dates or periods,36,40
2,Moscow,GPE,"Countries, cities, states",121,127
3,Dolgoprudny,GPE,"Countries, cities, states",132,143
4,Moscow Institute of Physics and Technology,ORG,"Companies, agencies, institutions, etc.",256,298
5,June 2026,DATE,Absolute or relative dates or periods,408,417
6,Obsidian,PRODUCT,"Objects, vehicles, foods, etc. (not services)",453,461
7,Two,CARDINAL,Numerals that do not fall under another type,680,683
8,Dolgoprudny,GPE,"Countries, cities, states",735,746
9,July 2026,DATE,Absolute or relative dates or periods,918,927


,entity,type
0,summer,DATE
1,2026,DATE
2,Moscow,GPE
3,Dolgoprudny,GPE
4,Moscow Institute of Physics and Technology,ORG
5,June 2026,DATE
6,Obsidian,PRODUCT
7,Two,CARDINAL
8,July 2026,DATE
9,Python,LANGUAGE


Именованные сущности — конкретные названия, даты и количества. Обычные слова friends, classmates и neural networks сами по себе именованными сущностями не являются. Английская статистическая модель может пропускать технические названия, поэтому финальный результат дополнен явно приведёнными правилами. В тексте есть место, даты и описание совместной работы с одногруппниками.